In [6]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_url


In [ ]:
repo_id = "open-index/arctic"
# Exact path file name structure used inside the huggingface repository 
file_subpath = "data/submissions/2024/2024_01_submissions.parquet"

print("Generating remote cloud URL...")
# 1. Manually resolve the cloud URL using Hugging Face Hub
data_url = hf_hub_url(
    repo_id=repo_id, 
    filename=file_subpath, 
    repo_type="dataset"
)

print(f"Streaming data directly from Hugging Face into memory...\n{data_url}")

# 2. Read the parquet file into memory chunks using Arrow
# Because it's a remote URL, pandas/pyarrow will stream it directly
target_subreddit = "economics"
cleaned_data = []

# Using chunks allows us to stream data instead of filling up memory
parquet_file = pd.read_parquet(data_url, engine="pyarrow")

print(f"Filtering rows specifically for r/{target_subreddit}...")
# Filter for our specific subreddit natively and efficiently
filtered_df = parquet_file[parquet_file['subreddit'].str.lower() == target_subreddit]

for _, post in filtered_df.iterrows():
    # Filter out empty or deleted posts
    if post.get("selftext") in ["[deleted]", "[removed]", ""] and post.get("title") in ["[deleted]", "[removed]", ""]:
        continue
        
    cleaned_data.append({
        "timestamp": pd.to_datetime(post.get("created_utc"), unit="s"),
        "title": post.get("title"),
        "text": post.get("selftext"),
        "score": post.get("score"),
        "num_comments": post.get("num_comments")
    })

if cleaned_data:
    df = pd.DataFrame(cleaned_data)
    df = df.sort_values(by="timestamp").reset_index(drop=True)
    df.to_csv("reddit_economics_2024_01.csv", index=False)
    print(f"\nSuccess! Successfully extracted {len(df)} posts into 'reddit_economics_2024_01.csv'.")
    print(df[['timestamp', 'title']].head())
else:
    print("\nNo economics posts found in this monthly snapshot.")


Generating remote cloud URL...
Streaming data directly from Hugging Face into memory...
https://huggingface.co/datasets/open-index/arctic/resolve/main/data/submissions/2024/2024_02_submissions.parquet


ArrowKeyError: A type extension with name pandas.period already defined